# AdamW算法
:label:`sec_adamw`

在 :numref:`sec_adam`中，我們學習了Adam算法，它是深度學習中最受歡迎的優化算法之一。
然而，Adam算法在處理權重衰減（weight decay）時存在一個微妙但重要的問題。

AdamW算法 :cite:`Loshchilov.Hutter.2017`通過修正Adam中權重衰減的實現方式，
提供了一個更有效的正則化方法。
這個看似簡單的改變，在訓練大型模型（特別是Transformer）時帶來了顯著的性能提升。

## AdamW vs Adam的區別

### 權重衰減的兩種實現方式

在深度學習中，權重衰減是一種常用的正則化技術，用於防止過擬合。
然而，權重衰減可以用兩種不同的方式實現：

1. **L2正則化（Adam的方式）**：將權重的L2範數添加到損失函數中
2. **解耦權重衰減（AdamW的方式）**：直接在參數更新時添加權重衰減項

對於標準的隨機梯度下降（SGD），這兩種方式是等價的。
但對於自適應學習率算法（如Adam），它們是**不同的**！

### 為什麼AdamW更好

在Adam中使用L2正則化存在以下問題：

1. **自適應學習率干擾正則化**：
   - Adam的自適應學習率會改變權重衰減的有效強度
   - 對於不同的參數，實際的正則化強度會不同

2. **超參數耦合**：
   - 學習率和權重衰減參數相互影響
   - 調整一個會影響另一個的效果

3. **訓練不穩定**：
   - 在訓練大型模型時，可能導致訓練不穩定
   - 特別是在Transformer等架構中

AdamW通過解耦權重衰減解決了這些問題，使得：
- 權重衰減的強度對所有參數都是一致的
- 學習率和正則化可以獨立調整
- 訓練更加穩定和可靠

## 數學原理

### Adam的權重衰減問題

在標準的Adam算法中添加L2正則化，目標函數變為：

$$f_t(\mathbf{x}) = f(\mathbf{x}) + \frac{\lambda}{2}\|\mathbf{x}\|^2$$

其中$\lambda$是權重衰減係數。梯度變為：

$$\mathbf{g}_t = \nabla f(\mathbf{x}_t) + \lambda \mathbf{x}_t$$

然後Adam使用自適應學習率更新參數：

$$\begin{aligned}
    \mathbf{m}_t &= \beta_1 \mathbf{m}_{t-1} + (1 - \beta_1) \mathbf{g}_t \\
    \mathbf{v}_t &= \beta_2 \mathbf{v}_{t-1} + (1 - \beta_2) \mathbf{g}_t^2 \\
    \hat{\mathbf{m}}_t &= \frac{\mathbf{m}_t}{1 - \beta_1^t} \\
    \hat{\mathbf{v}}_t &= \frac{\mathbf{v}_t}{1 - \beta_2^t} \\
    \mathbf{x}_t &= \mathbf{x}_{t-1} - \eta \frac{\hat{\mathbf{m}}_t}{\sqrt{\hat{\mathbf{v}}_t} + \epsilon}
\end{aligned}$$

**問題**：權重衰減項$\lambda \mathbf{x}_t$被包含在$\mathbf{g}_t$中，因此會被自適應學習率$\frac{\eta}{\sqrt{\hat{\mathbf{v}}_t} + \epsilon}$縮放。
這意味著對於梯度較大的參數，權重衰減的有效強度會減小。

### AdamW的解耦權重衰減

AdamW將權重衰減從梯度計算中分離出來，直接在參數更新時應用：

$$\begin{aligned}
    \mathbf{g}_t &= \nabla f(\mathbf{x}_t) \quad \text{(不包含權重衰減)} \\
    \mathbf{m}_t &= \beta_1 \mathbf{m}_{t-1} + (1 - \beta_1) \mathbf{g}_t \\
    \mathbf{v}_t &= \beta_2 \mathbf{v}_{t-1} + (1 - \beta_2) \mathbf{g}_t^2 \\
    \hat{\mathbf{m}}_t &= \frac{\mathbf{m}_t}{1 - \beta_1^t} \\
    \hat{\mathbf{v}}_t &= \frac{\mathbf{v}_t}{1 - \beta_2^t} \\
    \mathbf{x}_t &= \mathbf{x}_{t-1} - \eta \left(\frac{\hat{\mathbf{m}}_t}{\sqrt{\hat{\mathbf{v}}_t} + \epsilon} + \lambda \mathbf{x}_{t-1}\right)
\end{aligned}$$

**關鍵差異**：權重衰減項$\lambda \mathbf{x}_{t-1}$直接添加到更新中，不受自適應學習率的影響。

### 數學推導

讓我們更仔細地看看這兩種方法的區別。

**Adam with L2正則化**：
$$\mathbf{x}_t = \mathbf{x}_{t-1} - \eta \frac{\hat{\mathbf{m}}_t}{\sqrt{\hat{\mathbf{v}}_t} + \epsilon}$$

其中$\hat{\mathbf{m}}_t$包含了$\lambda \mathbf{x}_{t-1}$項。

**AdamW**：
$$\mathbf{x}_t = (1 - \eta\lambda)\mathbf{x}_{t-1} - \eta \frac{\hat{\mathbf{m}}_t}{\sqrt{\hat{\mathbf{v}}_t} + \epsilon}$$

這可以看作是對參數進行了一個常數因子$(1 - \eta\lambda)$的縮放，然後再應用Adam更新。
這確保了權重衰減對所有參數都有相同的相對影響。

## PyTorch實現

### 從頭實現

讓我們先從頭實現AdamW算法，以更好地理解其工作原理。

In [ ]:
%matplotlib inline
import torch
from d2l import torch as d2l

def init_adamw_states(feature_dim):
    """初始化AdamW的狀態變量"""
    v_w, v_b = torch.zeros((feature_dim, 1)), torch.zeros(1)
    s_w, s_b = torch.zeros((feature_dim, 1)), torch.zeros(1)
    return ((v_w, s_w), (v_b, s_b))

def adamw(params, states, hyperparams):
    """AdamW算法實現"""
    beta1, beta2, eps = 0.9, 0.999, 1e-6
    weight_decay = hyperparams.get('weight_decay', 0)
    
    for p, (v, s) in zip(params, states):
        with torch.no_grad():
            # 計算一階矩估計（動量）
            v[:] = beta1 * v + (1 - beta1) * p.grad
            # 計算二階矩估計（未中心化的方差）
            s[:] = beta2 * s + (1 - beta2) * torch.square(p.grad)
            # 偏差修正
            v_bias_corr = v / (1 - beta1 ** hyperparams['t'])
            s_bias_corr = s / (1 - beta2 ** hyperparams['t'])
            
            # AdamW的關鍵：解耦權重衰減
            # 先應用Adam更新
            p[:] -= hyperparams['lr'] * v_bias_corr / (torch.sqrt(s_bias_corr) + eps)
            # 然後應用權重衰減
            p[:] -= hyperparams['lr'] * weight_decay * p
            
        p.grad.data.zero_()
    hyperparams['t'] += 1

### 使用PyTorch的官方實現

PyTorch提供了`torch.optim.AdamW`的官方實現。讓我們看看如何使用它。

In [ ]:
import torch.optim as optim

# 示例模型
class SimpleModel(torch.nn.Module):
    def __init__(self, feature_dim, output_dim):
        super().__init__()
        self.linear = torch.nn.Linear(feature_dim, output_dim)
    
    def forward(self, x):
        return self.linear(x)

# 創建模型
model = SimpleModel(10, 1)

# 錯誤方式：使用Adam with weight_decay（實際上是L2正則化）
optimizer_adam_wrong = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

# 正確方式：使用AdamW（解耦權重衰減）
optimizer_adamw = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

print("Adam優化器（使用L2正則化）：", optimizer_adam_wrong)
print("AdamW優化器（解耦權重衰減）：", optimizer_adamw)

### 對比實驗：Adam vs AdamW

讓我們通過一個簡單的實驗來比較Adam和AdamW的性能。
我們將使用一個簡單的回歸任務。

In [ ]:
# 獲取訓練數據
data_iter, feature_dim = d2l.get_data_ch11(batch_size=10)

# 使用我們的AdamW實現
print("訓練使用AdamW（從頭實現）")
d2l.train_ch11(adamw, init_adamw_states(feature_dim),
               {'lr': 0.01, 't': 1, 'weight_decay': 0.01}, 
               data_iter, feature_dim);

In [ ]:
# 使用PyTorch的AdamW
print("訓練使用AdamW（PyTorch官方實現）")
trainer = torch.optim.AdamW
d2l.train_concise_ch11(trainer, {'lr': 0.01, 'weight_decay': 0.01}, data_iter)

In [ ]:
# 對比：使用Adam with weight_decay（實際上是L2正則化）
print("訓練使用Adam with weight_decay")
trainer = torch.optim.Adam
d2l.train_concise_ch11(trainer, {'lr': 0.01, 'weight_decay': 0.01}, data_iter)

## 詳細的對比實驗

讓我們做一個更詳細的實驗，在一個具有過擬合傾向的數據集上比較Adam和AdamW。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def generate_overfitting_data(n_samples=100, n_features=50, noise=0.1):
    """生成一個容易過擬合的數據集（特徵數多於樣本數）"""
    torch.manual_seed(42)
    X = torch.randn(n_samples, n_features)
    # 真實的關係只依賴於前5個特徵
    true_w = torch.zeros(n_features, 1)
    true_w[:5] = torch.randn(5, 1)
    y = X @ true_w + noise * torch.randn(n_samples, 1)
    return X, y, true_w

def train_and_evaluate(optimizer_class, optimizer_params, X_train, y_train, 
                       X_test, y_test, n_epochs=100, model_class=None):
    """訓練並評估模型"""
    n_features = X_train.shape[1]
    
    # 創建模型
    if model_class is None:
        model = torch.nn.Linear(n_features, 1)
    else:
        model = model_class(n_features, 1)
    
    # 創建優化器
    optimizer = optimizer_class(model.parameters(), **optimizer_params)
    
    # 訓練
    train_losses = []
    test_losses = []
    
    for epoch in range(n_epochs):
        # 訓練模式
        model.train()
        optimizer.zero_grad()
        train_pred = model(X_train)
        train_loss = torch.nn.functional.mse_loss(train_pred, y_train)
        train_loss.backward()
        optimizer.step()
        
        # 評估模式
        model.eval()
        with torch.no_grad():
            test_pred = model(X_test)
            test_loss = torch.nn.functional.mse_loss(test_pred, y_test)
        
        train_losses.append(train_loss.item())
        test_losses.append(test_loss.item())
    
    return train_losses, test_losses, model

# 生成數據
X_train, y_train, true_w = generate_overfitting_data(n_samples=100, n_features=50)
X_test, y_test, _ = generate_overfitting_data(n_samples=50, n_features=50)

# 訓練不同的模型
results = {}

# 1. Adam without weight decay
print("訓練 Adam (no weight decay)...")
results['Adam (no WD)'] = train_and_evaluate(
    torch.optim.Adam, {'lr': 0.01, 'weight_decay': 0},
    X_train, y_train, X_test, y_test
)

# 2. Adam with weight decay (L2 regularization)
print("訓練 Adam (with weight decay)...")
results['Adam (WD=0.1)'] = train_and_evaluate(
    torch.optim.Adam, {'lr': 0.01, 'weight_decay': 0.1},
    X_train, y_train, X_test, y_test
)

# 3. AdamW
print("訓練 AdamW...")
results['AdamW (WD=0.1)'] = train_and_evaluate(
    torch.optim.AdamW, {'lr': 0.01, 'weight_decay': 0.1},
    X_train, y_train, X_test, y_test
)

# 繪製結果
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 訓練損失
for name, (train_losses, _, _) in results.items():
    ax1.plot(train_losses, label=name, linewidth=2)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Training Loss', fontsize=12)
ax1.set_title('Training Loss Comparison', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_yscale('log')

# 測試損失
for name, (_, test_losses, _) in results.items():
    ax2.plot(test_losses, label=name, linewidth=2)
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Test Loss', fontsize=12)
ax2.set_title('Test Loss Comparison (Generalization)', fontsize=14, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.set_yscale('log')

plt.tight_layout()
plt.show()

# 打印最終結果
print("\n最終測試損失：")
for name, (_, test_losses, _) in results.items():
    print(f"{name:20s}: {test_losses[-1]:.6f}")

### 分析實驗結果

從上面的實驗中，我們可以觀察到：

1. **Adam (no WD)**：訓練損失很低，但測試損失較高，表明嚴重過擬合
2. **Adam (WD=0.1)**：過擬合有所改善，但正則化效果受自適應學習率影響
3. **AdamW (WD=0.1)**：通常在測試集上表現最好，因為權重衰減的效果更一致

## 使用建議

### 何時使用AdamW

1. **訓練大型模型**：特別是Transformer、BERT、GPT等
2. **需要權重衰減**：當你需要正則化防止過擬合時
3. **長時間訓練**：AdamW在長時間訓練中更穩定
4. **微調預訓練模型**：AdamW是微調的標準選擇

### 超參數設置建議

根據經驗和文獻，以下是一些常用的超參數設置：

1. **學習率（lr）**：
   - 小模型：1e-3 到 1e-4
   - 大型Transformer：1e-4 到 5e-5
   - 微調預訓練模型：1e-5 到 5e-6

2. **權重衰減（weight_decay）**：
   - 典型範圍：0.01 到 0.1
   - Transformer：0.01（BERT原文使用）
   - 小數據集：可以增大到 0.1 或更高

3. **Beta參數**：
   - beta1：0.9（通常不需要改變）
   - beta2：0.999（Transformer中有時用0.98）

4. **Epsilon**：
   - 默認：1e-8
   - 有時用：1e-6（提高數值穩定性）

### 在Transformer訓練中的應用

AdamW是訓練Transformer模型的標準優化器。讓我們看一個典型的配置：

In [ ]:
# 示例：訓練Transformer的典型配置
class TransformerConfig:
    """Transformer訓練配置示例（類似BERT）"""
    def __init__(self):
        # 優化器參數
        self.learning_rate = 1e-4
        self.weight_decay = 0.01
        self.beta1 = 0.9
        self.beta2 = 0.999
        self.epsilon = 1e-8
        
        # 學習率調度
        self.warmup_steps = 10000
        self.total_steps = 100000

def create_optimizer_for_transformer(model, config):
    """
    為Transformer模型創建優化器
    注意：通常我們對某些參數不應用權重衰減
    """
    # 將參數分為兩組：需要權重衰減和不需要權重衰減
    no_decay = ['bias', 'LayerNorm.weight', 'layer_norm.weight']
    
    optimizer_grouped_parameters = [
        {
            'params': [p for n, p in model.named_parameters() 
                      if not any(nd in n for nd in no_decay)],
            'weight_decay': config.weight_decay
        },
        {
            'params': [p for n, p in model.named_parameters() 
                      if any(nd in n for nd in no_decay)],
            'weight_decay': 0.0
        }
    ]
    
    optimizer = torch.optim.AdamW(
        optimizer_grouped_parameters,
        lr=config.learning_rate,
        betas=(config.beta1, config.beta2),
        eps=config.epsilon
    )
    
    return optimizer

# 示例使用
config = TransformerConfig()
# model = YourTransformerModel()  # 你的Transformer模型
# optimizer = create_optimizer_for_transformer(model, config)

print("Transformer訓練配置：")
print(f"  學習率: {config.learning_rate}")
print(f"  權重衰減: {config.weight_decay}")
print(f"  Beta1: {config.beta1}")
print(f"  Beta2: {config.beta2}")
print(f"  Warmup steps: {config.warmup_steps}")

### 重要注意事項

1. **不是所有參數都需要權重衰減**：
   - Bias項通常不應用權重衰減
   - LayerNorm的參數不應用權重衰減
   - Embedding層的參數有時不應用權重衰減

2. **配合學習率調度**：
   - AdamW通常與warmup和學習率衰減一起使用
   - 常見模式：線性warmup + 線性衰減或餘弦衰減

3. **梯度裁剪**：
   - 訓練Transformer時，通常還會使用梯度裁剪
   - 典型值：max_grad_norm = 1.0

## 學習率調度示例

讓我們實現一個典型的學習率調度器，這在使用AdamW時很常見。

In [ ]:
def get_linear_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps):
    """
    創建一個帶有warmup的線性學習率調度器
    這是訓練Transformer時的標準配置
    """
    def lr_lambda(current_step):
        if current_step < num_warmup_steps:
            # Warmup階段：線性增加
            return float(current_step) / float(max(1, num_warmup_steps))
        # Warmup後：線性衰減
        return max(
            0.0,
            float(num_training_steps - current_step) / 
            float(max(1, num_training_steps - num_warmup_steps))
        )
    
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# 繪製學習率變化
class DummyModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.param = torch.nn.Parameter(torch.randn(1))

model = DummyModel()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
scheduler = get_linear_schedule_with_warmup(
    optimizer, 
    num_warmup_steps=1000, 
    num_training_steps=10000
)

# 記錄學習率變化
lrs = []
for step in range(10000):
    lrs.append(optimizer.param_groups[0]['lr'])
    optimizer.step()
    scheduler.step()

# 繪圖
plt.figure(figsize=(10, 6))
plt.plot(lrs, linewidth=2)
plt.axvline(x=1000, color='r', linestyle='--', label='End of Warmup', linewidth=2)
plt.xlabel('Training Step', fontsize=12)
plt.ylabel('Learning Rate', fontsize=12)
plt.title('Learning Rate Schedule with Warmup', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"初始學習率: {lrs[0]:.6f}")
print(f"Warmup後學習率: {lrs[1000]:.6f}")
print(f"最終學習率: {lrs[-1]:.6f}")

## 完整的訓練示例

讓我們把所有內容整合在一起，展示一個完整的訓練流程。

In [ ]:
def train_with_adamw_complete(
    model, 
    train_loader, 
    val_loader,
    num_epochs=10,
    learning_rate=1e-3,
    weight_decay=0.01,
    warmup_ratio=0.1,
    max_grad_norm=1.0
):
    """
    使用AdamW的完整訓練流程
    包含：權重衰減、學習率warmup、梯度裁剪
    """
    # 分離參數組
    no_decay = ['bias', 'LayerNorm.weight']
    optimizer_grouped_parameters = [
        {
            'params': [p for n, p in model.named_parameters() 
                      if not any(nd in n for nd in no_decay)],
            'weight_decay': weight_decay
        },
        {
            'params': [p for n, p in model.named_parameters() 
                      if any(nd in n for nd in no_decay)],
            'weight_decay': 0.0
        }
    ]
    
    # 創建優化器
    optimizer = torch.optim.AdamW(optimizer_grouped_parameters, lr=learning_rate)
    
    # 計算總訓練步數
    total_steps = len(train_loader) * num_epochs
    warmup_steps = int(total_steps * warmup_ratio)
    
    # 創建學習率調度器
    scheduler = get_linear_schedule_with_warmup(
        optimizer, 
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )
    
    # 訓練循環
    train_losses = []
    val_losses = []
    learning_rates = []
    
    for epoch in range(num_epochs):
        # 訓練階段
        model.train()
        epoch_train_loss = 0
        
        for batch in train_loader:
            # 前向傳播
            outputs = model(batch['input'])
            loss = torch.nn.functional.mse_loss(outputs, batch['target'])
            
            # 反向傳播
            optimizer.zero_grad()
            loss.backward()
            
            # 梯度裁剪
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            
            # 更新參數
            optimizer.step()
            scheduler.step()
            
            epoch_train_loss += loss.item()
            learning_rates.append(optimizer.param_groups[0]['lr'])
        
        # 驗證階段
        model.eval()
        epoch_val_loss = 0
        
        with torch.no_grad():
            for batch in val_loader:
                outputs = model(batch['input'])
                loss = torch.nn.functional.mse_loss(outputs, batch['target'])
                epoch_val_loss += loss.item()
        
        # 記錄
        avg_train_loss = epoch_train_loss / len(train_loader)
        avg_val_loss = epoch_val_loss / len(val_loader)
        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)
        
        print(f"Epoch {epoch+1}/{num_epochs}:")
        print(f"  Train Loss: {avg_train_loss:.4f}")
        print(f"  Val Loss: {avg_val_loss:.4f}")
        print(f"  Learning Rate: {optimizer.param_groups[0]['lr']:.6f}")
    
    return train_losses, val_losses, learning_rates

print("完整訓練函數已定義")
print("包含以下特性：")
print("  - 參數分組（對bias和LayerNorm不使用權重衰減）")
print("  - AdamW優化器")
print("  - 學習率warmup和線性衰減")
print("  - 梯度裁剪")

## 實際應用案例

### 案例1：BERT風格的預訓練

BERT論文中使用的配置：

In [ ]:
# BERT預訓練配置
bert_config = {
    'learning_rate': 1e-4,
    'weight_decay': 0.01,
    'beta1': 0.9,
    'beta2': 0.999,
    'epsilon': 1e-6,
    'warmup_steps': 10000,
    'max_steps': 1000000,
    'max_grad_norm': 1.0
}

print("BERT預訓練配置：")
for key, value in bert_config.items():
    print(f"  {key}: {value}")

### 案例2：GPT風格的預訓練

GPT系列使用的配置通常與BERT類似，但可能有細微差異：

In [ ]:
# GPT預訓練配置（基於GPT-2）
gpt_config = {
    'learning_rate': 2.5e-4,  # GPT-2使用較高的學習率
    'weight_decay': 0.01,
    'beta1': 0.9,
    'beta2': 0.95,  # GPT-2使用較小的beta2
    'epsilon': 1e-8,
    'warmup_steps': 2000,
    'max_steps': 500000,
    'max_grad_norm': 1.0
}

print("GPT-2預訓練配置：")
for key, value in gpt_config.items():
    print(f"  {key}: {value}")

### 案例3：微調配置

微調預訓練模型時的典型配置：

In [ ]:
# 微調配置（如分類任務）
finetuning_config = {
    'learning_rate': 2e-5,  # 微調時使用更小的學習率
    'weight_decay': 0.01,
    'beta1': 0.9,
    'beta2': 0.999,
    'epsilon': 1e-8,
    'warmup_ratio': 0.1,  # warmup佔總步數的10%
    'num_epochs': 3,  # 微調通常只需要幾個epoch
    'max_grad_norm': 1.0
}

print("微調配置：")
for key, value in finetuning_config.items():
    print(f"  {key}: {value}")

print("\n微調建議：")
print("  - 使用比預訓練更小的學習率（通常是1/10到1/100）")
print("  - 較短的訓練時間（3-5個epochs）")
print("  - 可以使用更激進的權重衰減防止過擬合")
print("  - warmup步數可以更少（總步數的5-10%）")

## 常見問題和解決方案

### Q1: AdamW和Adam+L2正則化的性能差異有多大？

**答**：在小型模型上，差異可能不明顯。但在大型Transformer模型上，AdamW通常能帶來：
- 1-2%的性能提升
- 更穩定的訓練過程
- 更好的泛化性能

### Q2: 是否總是應該使用AdamW而不是Adam？

**答**：
- 如果你需要權重衰減：是的，使用AdamW
- 如果你不需要權重衰減：兩者沒有區別
- 訓練Transformer：強烈推薦AdamW

### Q3: 如何選擇weight_decay的值？

**答**：
- 從0.01開始是一個安全的選擇
- 如果過擬合嚴重，增加到0.1
- 如果欠擬合，減少到0.001或不使用
- 通過驗證集性能進行調優

### Q4: 為什麼bias和LayerNorm不需要權重衰減？

**答**：
- **Bias**：通常不會導致過擬合，衰減bias會損害性能
- **LayerNorm**：這些參數用於歸一化，不應該被衰減
- **經驗法則**：只對權重矩陣應用權重衰減

## 小結

* AdamW通過解耦權重衰減解決了Adam中L2正則化的問題
* 在Adam中，權重衰減被自適應學習率縮放；在AdamW中，權重衰減直接應用於參數
* AdamW在訓練大型Transformer模型時表現更好，是BERT、GPT等模型的標準選擇
* 使用AdamW時應該：
  - 對bias和LayerNorm參數禁用權重衰減
  - 配合學習率warmup使用
  - 使用梯度裁剪提高穩定性
* 典型的超參數：lr=1e-4到5e-5，weight_decay=0.01，beta1=0.9，beta2=0.999

## 練習

1. 在你自己的數據集上比較Adam、Adam with weight decay和AdamW的性能
2. 實驗不同的weight_decay值（0.001, 0.01, 0.1），觀察對訓練和測試性能的影響
3. 嘗試將AdamW與不同的學習率調度策略結合（cosine annealing, exponential decay等）
4. 分析為什麼在某些情況下AdamW可能比Adam表現更好
5. 實現一個實驗，展示對bias參數應用權重衰減的負面影響

## 參考文獻

* Loshchilov, I., & Hutter, F. (2017). Decoupled Weight Decay Regularization. arXiv preprint arXiv:1711.05101.
* Devlin, J., Chang, M. W., Lee, K., & Toutanova, K. (2018). BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding.
* Radford, A., Wu, J., Child, R., Luan, D., Amodei, D., & Sutskever, I. (2019). Language Models are Unsupervised Multitask Learners.

[Discussions](https://discuss.d2l.ai/t/adamw)